In [ ]:
# @title Install dependencies
!pip install -q transformers accelerate pandas tqdm sentencepiece

import os
import gc
import math
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Dict, Any, Optional

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# @title Configuration

# Choose your model:
# "sshleifer/distilbart-cnn-12-6"  -> much faster, slightly lower quality
# "facebook/bart-large-cnn"        -> slower, a bit better quality
MODEL_ID = "sshleifer/distilbart-cnn-12-6"  # @param ["sshleifer/distilbart-cnn-12-6", "facebook/bart-large-cnn"]

# Input CSV location (Option A: upload in next cell, Option B: set a path here)
CSV_PATH = ""  # @param {type:"string"}

# Column names
TITLE_COL = "title"    # @param {type:"string"}
TEXT_COL  = "raw_text" # @param {type:"string"}

# Output
OUTPUT_CSV = "summarized.csv"  # @param {type:"string"}
SAVE_EVERY = 100               # @param {type:"number"}  # save partial results every N rows

# Chunking
MAX_INPUT_TOKENS = 1024  # BART limit
CHUNK_OVERLAP = 50       # tokens of overlap between chunks to reduce boundary artifacts

# Generation parameters for chunk summaries
CHUNK_SUMMARY_KWARGS = {
    "max_length": 142,
    "min_length": 56,
    "do_sample": False,
    "num_beams": 4,
    "no_repeat_ngram_size": 3,
}

# Generation parameters for the final summary of merged chunk summaries
FINAL_SUMMARY_KWARGS = {
    "max_length": 200,
    "min_length": 80,
    "do_sample": False,
    "num_beams": 4,
    "no_repeat_ngram_size": 3,
}

# If you're tight on VRAM, you can try:
# - lowering num_beams
# - setting torch_dtype=torch.float16 (GPU only)
# - using distilbart
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else None


In [ ]:
# @title Load CSV (upload or from path)

from google.colab import files

if not CSV_PATH:
    print("Please upload your CSV (must contain columns:", TITLE_COL, "and", TEXT_COL, ")")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
    print("Using uploaded file:", CSV_PATH)

# If you want to use Google Drive instead, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = "/content/drive/MyDrive/your_file.csv"

df = pd.read_csv(CSV_PATH)
assert TITLE_COL in df.columns, f"Column '{TITLE_COL}' not found in CSV."
assert TEXT_COL in df.columns, f"Column '{TEXT_COL}' not found in CSV."

# Ensure text is string, handle NaNs
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[TITLE_COL] = df[TITLE_COL].fillna("").astype(str)

print("Rows:", len(df))
df.head()


In [ ]:
# @title Helper functions

def chunk_text_by_tokens(text: str,
                         tokenizer,
                         max_input_tokens: int = 1024,
                         overlap: int = 50) -> List[str]:
    """Chunk a long text by *tokens* to respect the model's max input size."""
    input_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(input_ids) <= max_input_tokens:
        return [text]

    chunks = []
    start = 0
    while start < len(input_ids):
        end = min(start + max_input_tokens, len(input_ids))
        chunk_ids = input_ids[start:end]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk_text)
        # move start forward with overlap
        start = end - overlap
        if start < 0:
            start = 0
    return chunks


def summarize_long_text(
    text: str,
    summarizer,
    tokenizer,
    max_input_tokens: int = 1024,
    chunk_overlap: int = 50,
    chunk_summary_kwargs: Optional[Dict[str, Any]] = None,
    final_summary_kwargs: Optional[Dict[str, Any]] = None,
) -> str:
    """Map-reduce summarization: chunk -> summarize chunks -> summarize merged chunks."""
    if not text.strip():
        return ""

    if chunk_summary_kwargs is None:
        chunk_summary_kwargs = {}
    if final_summary_kwargs is None:
        final_summary_kwargs = {}

    chunks = chunk_text_by_tokens(text, tokenizer,
                                  max_input_tokens=max_input_tokens,
                                  overlap=chunk_overlap)

    # If only one chunk, summarize directly
    if len(chunks) == 1:
        return summarizer(chunks[0], **chunk_summary_kwargs)[0]["summary_text"]

    # Otherwise, summarize each chunk, then summarize their concatenation
    partial_summaries = []
    for ch in chunks:
        out = summarizer(ch, **chunk_summary_kwargs)[0]["summary_text"]
        partial_summaries.append(out)

    merged = " ".join(partial_summaries)
    final = summarizer(merged, **final_summary_kwargs)[0]["summary_text"]
    return final


In [ ]:
# @title Load model & pipeline

print(f"Loading model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# device_map="auto" uses accelerate to put the model on GPU if available
# torch_dtype helps reduce VRAM usage on GPU
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if torch.cuda.is_available() else None
)

summarizer = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device_map="auto" if torch.cuda.is_available() else None
    # If you have a single GPU and want to force it:
    # device=0
)

print("Model loaded.")


In [ ]:
# @title Summarize all rows (with checkpoints & resume)

# If resuming a crashed/aborted run, try to read previous results:
if os.path.exists(OUTPUT_CSV):
    print(f"Found existing {OUTPUT_CSV}, attempting to resume...")
    df_out = pd.read_csv(OUTPUT_CSV)
    if "summary" in df_out.columns and len(df_out) == len(df):
        df["summary"] = df_out["summary"]
    else:
        # merge by index if shapes differ
        if "summary" not in df.columns:
            df["summary"] = None
        for i in range(min(len(df_out), len(df))):
            if pd.notna(df_out.loc[i, "summary"]) and (pd.isna(df.loc[i, "summary"]) or df.loc[i, "summary"] == ""):
                df.loc[i, "summary"] = df_out.loc[i, "summary"]
else:
    df["summary"] = None

start_idx = df["summary"].isna().idxmax() if df["summary"].isna().any() else len(df)
print(f"Starting/resuming from row: {start_idx}")

try:
    for i in tqdm(range(start_idx, len(df))):
        text = df.at[i, TEXT_COL]
        # summarize
        summary = summarize_long_text(
            text=text,
            summarizer=summarizer,
            tokenizer=tokenizer,
            max_input_tokens=MAX_INPUT_TOKENS,
            chunk_overlap=CHUNK_OVERLAP,
            chunk_summary_kwargs=CHUNK_SUMMARY_KWARGS,
            final_summary_kwargs=FINAL_SUMMARY_KWARGS
        )
        df.at[i, "summary"] = summary

        # periodic checkpoint
        if (i + 1) % SAVE_EVERY == 0:
            df.to_csv(OUTPUT_CSV, index=False)
            print(f"Checkpoint saved at row {i+1}")

    # final save
    df.to_csv(OUTPUT_CSV, index=False)
    print("Done! Saved to", OUTPUT_CSV)

except KeyboardInterrupt:
    print("Interrupted. Saving progress...")
    df.to_csv(OUTPUT_CSV, index=False)
    print("Partial results saved to", OUTPUT_CSV)
